In [ ]:
import os
import asyncio
import pandas as pd
from datetime import datetime, timedelta
from tinkoff.invest import (
    AsyncClient,
    Client,
    CandleInstrument,
    SubscriptionInterval,
    HistoricCandle,
    CandleInterval
)


Подгружаем остальные ноутбуки. Они вынесены в отдельные файлы, исходя из функциональности

In [2]:
# установка переменных среды
%run set_secrets.ipynb

In [3]:
# загрузка и сохранение в базу данных
%run store_db.ipynb


In [4]:
# Обработка данных 
%run process_data.ipynb

In [5]:
# Отображение котировок в дашборде
%run plot_data.ipynb

In [6]:
TOKEN = os.environ["TINKOFF_API_TOKEN"]

TABLE_NAME = "sber_stock"

In [ ]:
streaming_data = pd.DataFrame(columns=['time', 'open', 'high', 'low', 'close', 'volume', 'anomalies_price', 'anomalies_volume'])

In [8]:
# Поиск Figi (Financial Instrument Global Identifier) по тикеру
def figiByTicker(ticker):
    with Client(TOKEN) as client:
        instruments = client.instruments.shares()
        for instrument in instruments.instruments:
            if instrument.ticker == ticker:
                print(f"\nTicker {instrument.ticker}  {instrument.name} have figi={instrument.figi}\n")
                return instrument.figi
        return None

In [ ]:
# Функция-обработчик поступающих с биржи данных
# Преобразуем новую свечу в стандартный формат, сохраняем в общий DataFrame,
# Делаем проверку-очистку данных, добавление фичей и сохранение в postgresql

async def handle_streaming_data(candle: HistoricCandle):
    global streaming_data

    new_row = {
        'time': candle.time,
        'open': candle.open.units + candle.open.nano / 1e9,
        'high': candle.high.units + candle.high.nano / 1e9,
        'low': candle.low.units + candle.low.nano / 1e9,
        'close': candle.close.units + candle.close.nano / 1e9,
        'volume': candle.volume
    }
    print(f"Receive new candle: {new_row}")   
     
    if streaming_data.empty:
        streaming_data = pd.DataFrame([new_row])
    else:
        streaming_data = pd.concat([streaming_data, pd.DataFrame([new_row])], ignore_index=True)

    #streaming_data['time'] = pd.to_datetime(streaming_data['time'])
    
    #  Очистка и добавление фичей
    streaming_data = prepare_data(streaming_data)

    print(f"Create features for new row ok")   

    save_to_db(streaming_data.iloc[[-1]], TABLE_NAME)   
    print(f"Save to db new row ok")   
    


In [10]:
# Запрашиваем пропущенные данные 
def fetch_historical_data(figi, interval, start_date, end_date):
    with Client(TOKEN) as client:
        candles = []
        current_start = start_date
        while current_start < end_date:
            temp_end = current_start + timedelta(days=7)  
            if temp_end > end_date:
                temp_end = end_date
            try:
                _candles = client.get_all_candles(
                    figi=figi,
                    from_=current_start,
                    to=temp_end,
                    interval=interval
                )
                candles.extend(_candles)
                current_start = temp_end
            except Exception as e:
                print(f"Error fetching data from {current_start} to {temp_end}: {e}")
                break
        global streaming_data
        data = {
            'time': [candle.time for candle in candles],
            'open': [candle.open.units + candle.open.nano / 1e9 for candle in candles],
            'high': [candle.high.units + candle.high.nano / 1e9 for candle in candles],
            'low': [candle.low.units + candle.low.nano / 1e9 for candle in candles],
            'close': [candle.close.units + candle.close.nano / 1e9 for candle in candles],
            'volume': [candle.volume for candle in candles]
        }
        streaming_data = pd.concat([streaming_data if not streaming_data.empty else None, pd.DataFrame(data) ], ignore_index=True)

        return streaming_data

In [ ]:
def backfill_data(figi):
    global streaming_data
    max_depth=20  

    streaming_data = get_data(max_depth, TABLE_NAME)
    if streaming_data.empty:
        print(f"Not found saved records")
    else:
        last_record = streaming_data['time'].iloc[-1]
        print(f"Last record: {last_record}")

    if last_record:
        start_date = last_record + timedelta(hours=1)  
    else:
        start_date = datetime.now() - timedelta(days=max_depth)  
    end_date = datetime.now()

    if start_date < end_date:
        print(f"Backfilling data from {start_date} to {end_date}")
        historical_data = fetch_historical_data(figi, CandleInterval.CANDLE_INTERVAL_HOUR, start_date, end_date)
        print(f"Fetched {historical_data.shape[0]} rows")
        historical_data = clean_data(historical_data)
        historical_data = create_features(historical_data)
        save_to_db(historical_data, TABLE_NAME)
        print(f"Saved {historical_data.shape[0]} rows")
    else:
        print(f"Backfilling data not required")

In [12]:

async def start_streaming(figi):
    async with AsyncClient(TOKEN) as client:
        market_data_stream = client.create_market_data_stream()
        market_data_stream.candles.waiting_close().subscribe(  # Подписка на новые данные, приходят только в часы работы биржи)
            [
                CandleInstrument(
                    figi=figi,
                    interval=SubscriptionInterval.SUBSCRIPTION_INTERVAL_ONE_HOUR, 
                            # SUBSCRIPTION_INTERVAL_ONE_MINUTE 
                            # SUBSCRIPTION_INTERVAL_ONE_HOUR
                            # SUBSCRIPTION_INTERVAL_FIVE_MINUTES
                )
            ]
        )
        async for marketdata in market_data_stream:
            if marketdata.candle:
                await handle_streaming_data(marketdata.candle)


In [13]:
async def run_streaming():
    figi=figiByTicker("SBER") # VTBR
    print(f"Start load ticker {figi}\n")
    backfill_data(figi)  # Проверяем последние загуженные данные в базе, догружаем пропуски с предыдущего запуска
    await start_streaming(figi)

In [14]:
import threading

def run_dash_app():
    app.run(debug=False)

# Запуск Dash в отдельном потоке
threading.Thread(target=run_dash_app, daemon=True).start()

# http://127.0.0.1:8050/

In [15]:
loop = asyncio.get_event_loop()
asyncio.run_coroutine_threadsafe(run_streaming(), loop)


<Future at 0x7f7fa0e7fc40 state=pending>


Ticker SBER  Сбер Банк have figi=BBG004730N88

Start load ticker BBG004730N88

Last record: 2025-01-26 23:00:00
Backfilling data from 2025-01-27 00:00:00 to 2025-01-27 18:20:25.620516
Fetched 340 rows
Saved 340 rows
Receive new candle: {'time': datetime.datetime(2025, 1, 27, 15, 20, tzinfo=datetime.timezone.utc), 'open': 276.07, 'high': 276.1, 'low': 275.95, 'close': 276.05, 'volume': 1515}
Create features for new row ok
Save to db new row ok
Receive new candle: {'time': datetime.datetime(2025, 1, 27, 15, 21, tzinfo=datetime.timezone.utc), 'open': 276.05, 'high': 276.09, 'low': 276.02, 'close': 276.08, 'volume': 891}
Create features for new row ok
Save to db new row ok
